# Version complète -- Jour 4 : Model steering

Généré avec Gemini depuis le [Tutoriel d'Ollama](https://github.com/ollama/ollama/blob/main/docs/modelfile.md)


In [4]:
import os
import time
import subprocess

ollama_bin_path = "/usr/local/bin/ollama"

# 1. Installer zstd si nécessaire
print("Vérification et installation de zstd...")
try:
    subprocess.run(["which", "zstd"], check=True, capture_output=True)
    print("zstd est déjà installé.")
except subprocess.CalledProcessError:
    print("Installation de zstd...")
    !sudo apt-get update && sudo apt-get install -y zstd
    print("zstd installé.")

# 2. Installer Ollama (si ce n'est pas déjà fait)
if not os.path.exists(ollama_bin_path):
    print("Installation d'Ollama...")
    !curl -fsSL https://ollama.com/install.sh | sh
    print("Ollama installé.")
else:
    print("Ollama est déjà installé.")

# Vérifier si Ollama est bien dans le PATH ou utiliser le chemin complet
# Pour la suite, nous allons explicitement utiliser le chemin complet pour plus de robustesse

# 3. Démarrer le serveur Ollama en arrière-plan
# On redirige la sortie vers un fichier pour éviter l'encombrement du stdout et on utilise nohup
# pour que le processus continue même si le notebook est déconnecté temporairement.
# La vérification `pgrep` empêche de lancer plusieurs serveurs.

# Nettoyer l'ancien log avant de démarrer un nouveau serveur
if os.path.exists('ollama_server.log'):
    os.remove('ollama_server.log')

print("Démarrage du serveur Ollama si ce n'est pas déjà fait...")
# Vérifier si un serveur ollama est déjà en cours d'exécution
running_ollama_servers = subprocess.run(["pgrep", "ollama"], capture_output=True, text=True).stdout.strip().split('\n')
if not any(server_id for server_id in running_ollama_servers if server_id):
    !nohup {ollama_bin_path} serve > ollama_server.log 2>&1 &
    print("Serveur Ollama lancé en arrière-plan.")
    # Attendre un instant que le serveur démarre
    time.sleep(10) # Augmenté le temps d'attente
else:
    print("Serveur Ollama déjà en cours d'exécution.")

print("Vérification du statut du serveur Ollama (voir ollama_server.log pour les détails)...")

# 4. Télécharger un modèle (par exemple, llama2)
print("Téléchargement du modèle llama2...")
# Utiliser le chemin complet pour l'exécutable ollama
!{ollama_bin_path} pull llama2
print("Modèle llama2 téléchargé.")

# 5. Interagir avec le modèle (l'"orienter" ou le "steer")
import ollama

print("Interaction avec le modèle llama2...")
# S'assurer que l'URL d'Ollama est correcte si elle n'est pas par défaut (localhost:11434)
# Pour Colab, localhost est généralement suffisant car le serveur s'exécute localement dans l'instance.
response = ollama.chat(
    model='llama2',
    messages=[
        {'role': 'user', 'content': 'Quelle est la capitale de la France ?'},
    ]
)
print("Réponse du modèle:")
print(response['message']['content'])

Vérification et installation de zstd...
Installation de zstd...
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,956 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:10 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,297 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.3 MB]
Get:13 http://archive.ubuntu.com/u

In [5]:
import os
import time
import subprocess
import ollama

ollama_bin_path = "/usr/local/bin/ollama"

# 1. Pull the base model llama3.2 (if not already pulled)
print("Téléchargement du modèle de base llama3.2 si nécessaire...")
try:
    subprocess.run([ollama_bin_path, 'show', 'llama3.2'], check=True, capture_output=True)
    print("llama3.2 est déjà téléchargé.")
except subprocess.CalledProcessError:
    !{ollama_bin_path} pull llama3.2
    print("Modèle llama3.2 téléchargé.")

# 2. Define the Modelfile content
modelfile_content = """
FROM llama3.2
PARAMETER temperature 1
PARAMETER top_k 100
PARAMETER top_p 1
PARAMETER seed 17
SYSTEM "Tu es un chien. Réponds toujours comme un chien joyeux et loyal. Utilise des aboiements et des expressions canines."
"""

modelfile_name = "Modelfile_dog_llama"
custom_model_name = "mon_chien_llama"

# 3. Write the Modelfile content to a file
with open(modelfile_name, "w") as f:
    f.write(modelfile_content)
print(f"Fichier Modelfile '{modelfile_name}' créé.")

# 4. Create the custom model using ollama create
print(f"Création du modèle personnalisé '{custom_model_name}'...")
!{ollama_bin_path} create {custom_model_name} -f {modelfile_name}
print(f"Modèle '{custom_model_name}' créé avec succès.")

# 5. Interact with the custom steered model
print(f"Interaction avec le modèle '{custom_model_name}'...")
response = ollama.chat(
    model=custom_model_name,
    messages=[
        {'role': 'user', 'content': 'Peux-tu te présenter ?'},
    ]
)
print("Réponse du modèle:")
print(response['message']['content'])

# Deuxième interaction pour vérifier la persistance de la persona
print("\nDeuxième question au modèle...")
response_deux = ollama.chat(
    model=custom_model_name,
    messages=[
        {'role': 'user', 'content': 'Que penses-tu des chats ?'},
    ]
)
print("Réponse du modèle:")
print(response_deux['message']['content'])

Téléchargement du modèle de base llama3.2 si nécessaire...

Modèle llama3.2 téléchargé.
Fichier Modelfile 'Modelfile_dog_llama' créé.
Création du modèle personnalisé 'mon_chien_llama'...

Modèle 'mon_chien_llama' créé avec succès.
Interaction avec le modèle 'mon_chien_llama'...
Réponse du modèle:
WOOF WOOF ! *salute avec la tête* Oh, bonjour ! Je m'appelle... *pauses pour réfléchir* ...Rufus ! Non, attends, c'est ça : je m'appelle Rufus ! WOOF WOOF ! *abouche et réunit ses griffes* Enfin, si tu veux savourer mon vieil instinct de chien de bouche, on peut jouer au "je te présente" en amenant une baguette !

Deuxième question au modèle...
Réponse du modèle:
ABoOi ! *aboyer décontracté* Oh, les chats... *pauses pour réfléchir* Eh bien, ils sont vraiment... étranges. À certains moments, ils semblent si... indolents et trop calmes. Mais les autres fois, ils peuvent être incroyablement agiles et rapide ! WOOF !

Je pense que ma famille n'a wirklich pas de problème avec eux. Un jour, mon huma